In [424]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
#

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_5min.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_5min_1e6.nc') #***
res='1km';t_res='5min'
Np_str='1e6'

# # dx = 1km; Np = 50M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
# res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# parcel=parcel.isel(time=np.arange(0,400+1))

In [425]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [426]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=60 #150 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(parcel['xh']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=1
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 0, end_job = 16667


In [427]:
#Indexing Array with JobArray
parcel=parcel.isel(xh=slice(start_job,end_job))
#(for 150_000_000 parcels use 500-1000 jobs)

In [428]:
# Reading Back Data Later
##############
def make_data_dict(var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:,start_job:end_job] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:,start_job:end_job].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [429]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g', 'A_c', 'Z', 'Y', 'X']
data_dict = make_data_dict(var_names,read_type)
A_g, A_c, Z, Y, X = (data_dict[k] for k in var_names)
# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory(globals())

Top 10 objects with highest memory usage
{'X': '17.73 MB', 'Y': '17.73 MB', 'Z': '17.73 MB', 'A_g': '17.73 MB', 'A_c': '17.73 MB', 'zero_histogram': '8.0 MB', 'A': '7.09 MB', 'QV': '3.55 MB', 'TH': '3.55 MB', 'VAR1': '3.55 MB'}

0.11439 GB in use overall


In [430]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'VARS_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['QV','TH']
data_dict = make_data_dict(var_names,read_type)
QV, TH = (data_dict[k] for k in var_names)
check_memory(globals())

Top 10 objects with highest memory usage
{'X': '17.73 MB', 'Y': '17.73 MB', 'Z': '17.73 MB', 'A_g': '17.73 MB', 'A_c': '17.73 MB', 'QV': '8.87 MB', 'TH': '8.87 MB', 'zero_histogram': '8.0 MB', 'A': '7.09 MB', 'VAR1': '3.55 MB'}

0.12503 GB in use overall


In [431]:
##########################
#SETUP SECTION

In [432]:
zfs=data['zf'].data*1000
times=(data['time'].data / 1e9).astype(int)
Nt=len(data['time'])

In [433]:
##########################
#BUILDING HISTOGRAM FUNCTIONS

In [536]:
def SetupHistSampling(histz_vals, histt_vals):
    # Process z levels
    histz_str = np.array([str(z) for z in histz_vals])
    histz = np.clip(np.searchsorted(zfs, histz_vals) - 1, 0, None)

    # Process t levels
    histt_str = np.array([str(t) for t in histt_vals])
    histt_idx = np.clip(np.searchsorted(times, histt_vals) - 1, 0, None)
    histt = [np.arange(0, ind + 2) for ind in histt_idx]

    return histz, histt, histt_str, histz_str

In [538]:
#INITIALIZE ALL HISTOGRAMS IN A DICTIONARY
def MakeDictionary(histz_str,histt_str,hist_size):
    
    # Create a 2D zero-initialized array (this is your histogram storage array)
    zero_histogram = np.zeros(hist_size,dtype=int)
    
    # Initialize a dictionary to store the histograms
    Histograms = {}
    #store each histogram in a dictionary
    for z in histz_str:
        for t in histt_str:        
            Histograms[f"{z}_{t}"] = np.copy(zero_histogram)
    
    return Histograms
# hist_size = (1000, 1000)  
# Dictionary = MakeDictionary(histz_str,histt_str,hist_size)

In [611]:
#*#*
def MakeHistogram(VAR1s,VAR2s):
    ###GIVEN TWO FlATTENED ARRAYS 
    ###WILL BIN EACH VALUE IN 2D HISTOGRAM
    Output=np.ones((1000,1000),dtype=int) #*#*
    return Output

In [540]:
##########################
#BUILDING ALGORITHM FUNCTIONS

In [541]:
class BreakException(Exception):
    pass
def BREAK(var=None):
    print(var)
    raise BreakException 

#GET LAGRANGIAN SPATIAL ARRAYS
def GetSpace(t, p):#, X, Y, Z):
    z = Z[t, p]; zs = Z[t] 
    y = Y[t, p]; ys = Y[t]
    x = X[t, p]; xs = X[t]
    return z,y,x,zs,ys,xs

def SpaceConditional(z,y,x,D,zs,ys,xs):
    cond1 = D==1; cond2 = zs==z; 
    cond3 = ys==y; cond4 = xs==x
    #where the entrained parcels are in same grid box
    where=cond1&cond2&cond3&cond4 
    return where

#####
def Meshgrid_Advanced(ts, ps):
    t_grid, p_grid = np.meshgrid(ts, ps)
    return t_grid.flatten(), p_grid.flatten()

def SubsetZ(histz,t_lst,p_lst):
    Zs=Z[t_lst,p_lst]
    where_z = np.where(np.isin(Zs,histz))[0]
    t_lst=t_lst[where_z];p_lst=p_lst[where_z]
    return t_lst,p_lst

In [621]:
histt

[array([0, 1]),
 array([0, 1, 2]),
 array([0, 1, 2]),
 array([0, 1, 2, 3]),
 array([0, 1, 2, 3, 4])]

In [635]:
#FIND ENTRAINMENT PARCELS
def GetEntrainmentHistory(A,VAR1,VAR2,Dictionary,t,p):
    if t<histt[-1][-1]:
        return np.zeros_like(Dictionary)

    #marking entrained parcels
    D=A[t] - A[t-1]
    D=np.where(D>0,True,False)

    #getting some spatial variables
    [z,y,x,zs,ys,xs] = GetSpace(t,p)
    if z not in histz:
        return np.zeros_like(Dictionary)
    else:
        z_string=histz_str[histz == z][0]
    
    #locating entrained parcels 
    where = SpaceConditional(z,y,x,D,zs,ys,xs)
    ps=np.where(where)

    #get trackback trajectories
    for (ts,t_string) in zip(histt,histt_str):
        #meshgrid calculations 
        minus_ts=(t - ts)[(t - ts) < Nt]
        [t_flat,p_flat] = Meshgrid_Advanced(minus_ts, ps)

        #get variable data along trajectory
        VAR1s = VAR1[t_flat,p_flat]
        VAR2s = VAR2[t_flat,p_flat]

        #ADD HISTOGRAM TO A DICTIONARY
        Histogram = MakeHistogram(VAR1s,VAR2s)
        Dictionary[f'{z_string}_{t_string}'] += Histogram

    #return the updated dictionary
    return Dictionary
        

In [573]:
##########################
#LOADING DATA

In [574]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

# Reading Back Data Later
##################################################################
#DOMAIN SUBSETTING
def DOMAIN_SUBSET(out_arr):
    print(f'length before: {len(out_arr)}')

    #SETTING UP
    ###########
    ocean_percent=2/8
    left_to_coast=data['xh'][0]+(data['xh'][-1]-data['xh'][0])*ocean_percent
    
    where_coast_xh=np.where(data['xh']>=left_to_coast)[0][0]#-25
    where_coast_xf=np.where(data['xf']>=left_to_coast)[0][0]#-25
    end_xh=len(data['xh'])-1-50
    end_xf=len(data['xf'])-1-50
    
    print(f'x in {0}:{where_coast_xh-1} FOR SEA')
    print(f'x in {where_coast_xh}:{end_xh} FOR LAND')
    
    t_start=36 
    t_end=len(data['time']) # if res=='250m': t_end=410
    print(f't in {t_start}:end (8 hours)')

    #SUBSETTING CODE
    ################
    t,p=out_arr[:,1],out_arr[:,0]
    if 'job_array' in globals():
        p -= index_adjust

    #GETTING X VALUES OF EACH PARCEL 
    fancy_index=True
    # fancy_index=False
    if fancy_index==True:
        xs=X[t,p] #FANCY INDEXING
    elif fancy_index==False: #(slightly longer, but avoids loading X into memory)
        with h5py.File(in_file, 'r') as f:
            xs=[]
            for i, j in zip(t, p):
                xs.append(f['X'][i, j])
            xs=np.array(xs)
    ################
    
    out_arr=out_arr[np.where((xs>=where_coast_xh)&(xs<=end_xh))]
    out_arr=out_arr[np.where(out_arr[:,1]<=t_end)]

    print(f'==> length after: {len(out_arr)}'+'\n')
    return out_arr

In [575]:
#LOADING BACK IN
def load_file():
    in_file=dir+f'Project_Algorithms/Tracking_Algorithms/parcel_tracking_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(in_file, 'r') as hf:
        out_arr=hf['out_arr'][:]
        save_arr=hf['save_arr'][:]
        save2_arr=hf['save2_arr'][:]
    return out_arr,save_arr,save2_arr
[out_arr,save_arr,save2_arr]=load_file()

print('list of first 10 ignored parcels');
print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels'+'\n')

if 'job_array' in globals():
    #APPLYING JOB_ARRAY TO PARCEL NUMBER
    ####################################
    def job_filter(arr):
        return arr[(arr[:,0]>=start_job)&(arr[:,0]<end_job)]
    print('Applying Job Array')
    out_arr=job_filter(out_arr)
    save_arr=job_filter(save_arr)

print(f'there are a total of {len(out_arr)} CL parcels and {len(save_arr)} nonCL parcels'+'\n')

#CHOOSING UNIQUE INDEXES
###############################################################################
def remove_duplicates(arr):
    lst = []
    unique_values, counts = np.unique(arr[:, 0], return_counts=True)
    duplicates = unique_values[counts > 1]
    for elem in duplicates:
        idx = np.where(arr[:, 0] == elem)[0]
        extras = idx[np.where(arr[idx, 1] != np.min(arr[idx, 1]))]
        lst.extend(extras)
    mask = np.ones(len(arr), dtype=bool)
    mask[lst] = False
    return arr[mask]
out_arr=remove_duplicates(out_arr)
save_arr=remove_duplicates(save_arr)
###############################################################################

############################################################
#SUBSETTING
subset=True
if subset==True:
    out_arr=DOMAIN_SUBSET(out_arr)
    save_arr=DOMAIN_SUBSET(save_arr)
############################################################

ALL_out_arr=out_arr.copy(); ALL_save_arr=save_arr.copy()

list of first 10 ignored parcels
there are a total of 14754 CL parcels and 14545 nonCL parcels

Applying Job Array
there are a total of 45 CL parcels and 56 nonCL parcels

length before: 45
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 45

length before: 56
x in 0:127 FOR SEA
x in 128:461 FOR LAND
t in 36:end (8 hours)
==> length after: 41



In [576]:
#...... more data loading

In [577]:
##########################
#STARTING UP

In [628]:
#SET PARAMETERS HERE
type='general'; # type='cloudy'
ARR=ALL_out_arr.copy()
#HISTOGRAM SETUP PARAMETERS
histz_vals=[200,500,1000] #unit: meters
histt_vals=[200,400,600,800,1000] #unit:seconds
hist_size = (1000, 1000)  

In [629]:
#INITIALIZE VARIABLES
A = A_c.copy() if type == 'cloudy' else A_g.copy()
VAR1=QV.copy();VAR2=TH.copy()
t_lst=ARR[:,1]; p_lst=ARR[:,0]
[t_lst,p_lst]=SubsetZ(histz,t_lst,p_lst)#SUBSETTING DATA FOR Zs in histz (FOR FASTER SPEED)

#HISTOGRAM INFO
[histz, histt, histt_str, histz_str] = SetupHistSampling(histz_vals, histt_vals)
print(histz,histt)

#INITIALIZE HISTOGRAMS
Dictionary = MakeDictionary(histz_str,histt_str,hist_size)

[2 4 6] [array([0, 1]), array([0, 1, 2]), array([0, 1, 2]), array([0, 1, 2, 3]), array([0, 1, 2, 3, 4])]


In [630]:
##########################
#RUNNING

In [636]:
for (t,p) in zip(t_lst,p_lst):
    Dictionary = GetEntrainmentHistory(A,VAR1,VAR2,Dictionary,t,p)
    # BREAK()

In [637]:
Dictionary

{'200_200': array([[16, 16, 16, ..., 16, 16, 16],
        [16, 16, 16, ..., 16, 16, 16],
        [16, 16, 16, ..., 16, 16, 16],
        ...,
        [16, 16, 16, ..., 16, 16, 16],
        [16, 16, 16, ..., 16, 16, 16],
        [16, 16, 16, ..., 16, 16, 16]]),
 '200_400': array([[8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        ...,
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8]]),
 '200_600': array([[8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        ...,
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8]]),
 '200_800': array([[8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        ...,
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 8, 8, 8]]),
 '200_1000': array([[8, 8, 8, ..., 8, 8, 8],
        [8, 8, 8, ..., 